## 一、Series 中括号 [] 操作类型


### Series 索引操作对照表

|       操作类型       |            示例            |         返回结果          |                   说明                   |
|:----------------:|:------------------------:|:---------------------:|:--------------------------------------:|
|     *单标签索引*      |       `sr['fans']`       |          标量值          |               按标签获取单个元素                |
|     *多标签列表*      | `sr[['fans', 'likes']]`  |       子 Series        |               按多个标签获取子集                |
|     *单位置索引*      |         `sr[0]`          |          标量值          |               按整数位置获取元素                |
|     *多位置列表*      |       `sr[[1, 3]]`       |       子 Series        |               按多个位置获取子集                |
|      *位置切片*      |        `sr[0:3]`         |       子 Series        |              左闭右开（按位置切片）               |
|      *标签切片*      |      `sr['a':'c']`       |       子 Series        |              包含两端（需索引有序）               |
|      *布尔索引*      |       `sr[sr > 0]`       |       子 Series        |               通过布尔数组筛选元素               |


## 二、布尔索引核心逻辑

#### 本质：通过布尔值数组（True/False）筛选数据。
#### 规则：
- 布尔数组长度必须与数据行数一致。
- True 保留该行/元素，False 丢弃。
#### 多条件组合：
```
# 与操作
df[(df['fans'] > 0) & (df['likes'] < 100)]
# 或操作
df[(df['fans'] == 0) | (df['likes'] == 0)]
# 非操作
df[~(df['fans'] == 0)]
```

## 三、视图（View） vs 副本（Copy）

| 特性	         | 视图（View）                  | 	副本（Copy）        |
|-------------|---------------------------|------------------|
| 内存共享	       | 是（修改影响原数据）	               | 否（独立内存）          |
| 创建方式	       | sr = df['fans']	          | sr = df['fans'].copy() |
| 修改风险	       | 可能意外修改原数据	                | 安全操作             |
#### 验证方法
```
# 检查是否共享内存
print(sr.values.base is df['fans'].values.base)  # True=视图，False=副本
```

## 四、安全赋值操作

#### 直接修改原数据：
```
df.loc[df['fans'] == 0, 'fans'] = df['fans'].mean()  # 使用 .loc 避免警告
```
#### 创建独立副本修改：
```
sr_copy = df['fans'].copy()
sr_copy[sr_copy == 0] = sr_copy.mean()
```


## 五、.loc 和 .iloc 的精准索引

|方法	|索引类型	|语法示例	|特点|
|-------|--------|---------|-------|
|.loc	|标签索引	|df.loc['行标签', '列标签']	|切片包含两端|
|.iloc	|位置索引	|df.iloc[0:5, 2]	|切片左闭右开|
```
# 按标签筛选行和列
df.loc['推文1':'推文114', 'fans']

# 按位置筛选行和列
df.iloc[0:114, df.columns.get_loc('fans')]
```


## 六、常见错误与解决方案

#### 链式赋值警告：
```
# ❌ 错误写法（触发警告）
df['fans'][df['fans'] == 0] = mean_value
# ✅ 正确写法
df.loc[df['fans'] == 0, 'fans'] = mean_value
```
#### 误用副本修改：
```
sr = df['fans']          # 可能是视图
sr = sr[sr > 0]          # 生成新副本
sr[0] = 100              # 修改不影响原数据
```

 ## 七、最佳实践总结
####  明确操作目标：
- 修改原数据 → 使用 .loc 直接操作。
- 保留原数据 → 创建副本 .copy()。
#### 避免链式索引：
-  用 .loc 替代 df[][]] 分步操作。
-  验证索引类型：
-  检查是视图还是副本，避免意外修改。

## 八、速查表


|场景	| 代码模板                      |
|------|---------------------------|
|筛选符合条件的行	| df[df['列名'] > 值]          |
|多列筛选	| df.loc[:, ['列1', '列2']]   |
|修改指定条件的数据	| df.loc[条件, 列名] = 新值       |
|创建安全副本	| sr_copy = df['列名'].copy() |
